In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# Episode-based training framework

class Episode:
    """Represents a single N-way K-shot learning episode"""
    def __init__(self, support_data: torch.Tensor, support_labels: torch.Tensor,
                 query_data: torch.Tensor, query_labels: torch.Tensor):
        self.support_data = support_data
        self.support_labels = support_labels
        self.query_data = query_data
        self.query_labels = query_labels
        
    @property
    def n_way(self) -> int:
        return len(torch.unique(self.support_labels))
    
    @property
    def k_shot(self) -> int:
        return len(self.support_data) // self.n_way
    
    @property
    def n_query(self) -> int:
        return len(self.query_data) // self.n_way


class EpisodeSampler:
    """Samples episodes for few-shot learning"""
    
    def __init__(self, data: np.ndarray, labels: np.ndarray, 
                 n_way: int = 5, k_shot: int = 5, n_query: int = 15):
        self.data = data
        self.labels = labels
        self.n_way = n_way
        self.k_shot = k_shot
        self.n_query = n_query
        
        # Create class-wise data indices
        self.class_indices = {}
        unique_labels = np.unique(labels)
        for label in unique_labels:
            self.class_indices[label] = np.where(labels == label)[0]
            
    def sample_episode(self) -> Episode:
        """Sample a single N-way K-shot episode"""
        # Randomly select N classes
        selected_classes = np.random.choice(
            list(self.class_indices.keys()), 
            size=self.n_way, 
            replace=False
        )
        
        support_data = []
        support_labels = []
        query_data = []
        query_labels = []
        
        # For each selected class
        for i, class_label in enumerate(selected_classes):
            # Get indices for this class
            class_idx = self.class_indices[class_label]
            
            # Randomly sample K+Q examples
            selected_idx = np.random.choice(
                class_idx, 
                size=self.k_shot + self.n_query, 
                replace=False
            )
            
            # Split into support and query
            support_idx = selected_idx[:self.k_shot]
            query_idx = selected_idx[self.k_shot:]
            
            # Add to episode
            support_data.extend(self.data[support_idx])
            support_labels.extend([i] * self.k_shot)  # Use relative labels (0 to N-1)
            
            query_data.extend(self.data[query_idx])
            query_labels.extend([i] * self.n_query)
            
        # Convert to tensors
        support_data = torch.FloatTensor(support_data)
        support_labels = torch.LongTensor(support_labels)
        query_data = torch.FloatTensor(query_data)
        query_labels = torch.LongTensor(query_labels)
        
        return Episode(support_data, support_labels, query_data, query_labels)
    
    def generate_episodes(self, num_episodes: int) -> List[Episode]:
        """Generate multiple episodes"""
        return [self.sample_episode() for _ in range(num_episodes)]
